# PetGuard AI – Final Build

Continues midterm pretrained pet detection into **video → zone decision → temporal event → annotated video / CSV**.

Run on Google Colab. The scene-level temporal confirmation is a *single-zone prototype*, not per-pet tracking.

**Important:** This notebook is a ready-to-run starter, not a claim that the model meets its metric targets.

## 1. Clone the existing midterm repository and install dependencies

In [ ]:
!git clone -q https://github.com/iamhuongthn/PetGuard-AI.git /content/PetGuard-AI || true
%cd /content/PetGuard-AI
!pip -q install ultralytics opencv-python-headless numpy pandas matplotlib


## 2. Load the final implementation modules

The ZIP starter has `src/` modules. Before running this notebook, upload the ZIP archive to Colab using the next cell. This will install the `src/` modules in the cloned repository; no GitHub repository changes are made yet.


In [ ]:
from google.colab import files
import zipfile, pathlib, shutil
archive=files.upload()
zname=next(iter(archive))
with zipfile.ZipFile(zname) as z:
    z.extractall("/content")
source=pathlib.Path("/content/PetGuard-AI-Final/src")
assert source.exists(), "Upload PetGuard_AI_Final_Colab_Starter.zip"
shutil.copytree(source, "/content/PetGuard-AI/src", dirs_exist_ok=True)
print("Final modules installed")


In [ ]:
import sys, pathlib, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Video
from google.colab import files
sys.path.insert(0,'/content/PetGuard-AI')
from src.main import process_video
from src.zone import resize_frame
from ultralytics import YOLO
print('Source modules loaded')


## 3. Upload a properly licensed input MP4

In [ ]:
uploaded=files.upload()
assert uploaded, 'Select a video file'
name=next(iter(uploaded))
video_dir=pathlib.Path('/content/PetGuard-AI/samples/videos');video_dir.mkdir(parents=True,exist_ok=True)
video_path=video_dir/pathlib.Path(name).name
pathlib.Path(name).replace(video_path)
cap=cv2.VideoCapture(str(video_path))
assert cap.isOpened(), 'Video could not be opened'
fps=cap.get(cv2.CAP_PROP_FPS)
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(video_path, 'FPS:',fps,'frames:',total)


## 4. Review sample frames and select a polygon

In [ ]:
TARGET_WIDTH=960
fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,frac in zip(axes,[0.1,0.5,0.9]):
    cap.set(cv2.CAP_PROP_POS_FRAMES,max(0,int(total*frac)))
    ret,frame=cap.read()
    if not ret:continue
    resized=resize_frame(frame,TARGET_WIDTH)
    ax.imshow(cv2.cvtColor(resized,cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {int(total*frac)}')
    ax.set_xlim(0,resized.shape[1]);ax.set_ylim(resized.shape[0],0)
    ax.set_xticks(np.arange(0,resized.shape[1],100));ax.set_yticks(np.arange(0,resized.shape[0],100))
    ax.grid(alpha=.25)
cap.release();plt.show()


## 5. Configure the zone and model

**Edit POLYGON after viewing the frames above.** Points use the 960-pixel-wide resized image coordinates. For floor zones use `floor`. For couch zones consider `overlap` (2D proxy, not true pet-on-couch recognition).

In [ ]:
# EXAMPLE ONLY; MUST EDIT for the actual video
POLYGON=[(350,200),(760,200),(760,500),(350,500)]
ZONE_MODE='floor'           # or 'overlap' for furniture
WEIGHTS='yolov8s.pt'        # based on one-frame comparison; validate on more frames
CONFIDENCE=0.35
IMG_SIZE=960


## 6. Run the full video pipeline

In [ ]:
out_dir=pathlib.Path('/content/PetGuard-AI/results')
out_dir.mkdir(parents=True,exist_ok=True)
summary=process_video(video_path,out_dir/'petguard_annotated.mp4',out_dir/'events.csv',
                      polygon=POLYGON,weights=WEIGHTS,target_width=TARGET_WIDTH,
                      confidence=CONFIDENCE,imgsz=IMG_SIZE,mode=ZONE_MODE,confirm_frames=5)
print(summary)
print(pd.read_csv(summary['csv']).head())


## 7. Convert MP4 for browser playback and preview (if ffmpeg is available)

In [ ]:
import subprocess
browser_video=out_dir/'petguard_browser.mp4'
cmd=['ffmpeg','-y','-loglevel','error','-i',str(summary['output']),'-c:v','libx264','-pix_fmt','yuv420p',str(browser_video)]
subprocess.run(cmd,check=True)
display(Video(str(browser_video),embed=True,width=700))


## 8. Save outputs to your computer

In [ ]:
from google.colab import files
files.download(str(browser_video))
files.download(summary['csv'])
files.download(summary['predictions_csv'])


## 9. Final evaluation (requires manual ground truth)

Do **not** treat model confidence or `frame_predictions.csv` as accuracy. Label independent held-out examples first, then compare predictions.

Keep Oxford-IIIT (100), COCO (50), and PetGuard-specific (50) as the midterm plan, recording any justified changes. Use IoU-based one-to-one matching for detection recall and a separate SAFE/VIOLATION confusion matrix for zone results.

In [ ]:
# Paste verified counts from your labeled PetGuard-specific validation set.
TP=TN=FP=FN=0
n=TP+TN+FP+FN
if n==0:
    print('No labeled evaluation entered yet; results not measured.')
else:
    print('Zone accuracy:',(TP+TN)/n)
    print('False violation rate:', FP/(FP+TN) if FP+TN else 'undefined')
    print('Violation recall:', TP/(TP+FN) if TP+FN else 'undefined')
